# Recommendation analysis

In [1]:
import torch
import sys

from recbole.utils.utils import init_seed
from recbole.utils.logger import init_logger
from logging import getLogger
from recbole.data.utils import create_dataset
from recbole.data.utils import data_preparation
from recbole.utils.utils import get_model
import pandas as pd
import ast
import json

# Oszaleje kurwa
class WandbMock:
    def init(self, *args, **kwargs): return None
    def log(self, *args, **kwargs): return None

sys.modules['wandb'] = WandbMock()

In [2]:
MODELS = ["base.pth", "cat_titles.pth", "images.pth"]

### Import saved models

In [3]:
models = []
for model_name in MODELS:
    data = torch.load(f"../../../../../saved/{model_name}", map_location='cpu', weights_only=False)

    config = data['config']
    config.final_config_dict['data_path'] = r'\\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\Amazon_Sports_and_Outdoors'
    config['device'] = 'cpu'
    config['wandb_run_name'] = 'test_run'
    config['orig_attribute_hidden_size'] = [420, 420]
    config['eval_batch_size'] = 1

    for attr, value in config.__dict__.items():
        print(f"{attr} = {value}")

    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    dataset = create_dataset(config)
    logger.info(dataset)

    train_data, valid_data, test_data = data_preparation(config, dataset)
    model1 = get_model(config['model'])(config, train_data.dataset).to(config['device'])
    logger.info(model1)

    model1.load_state_dict(data['state_dict'])
    model1.eval()
    models.append(model1)

25 Jan 20:15    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 212
state = INFO
reproducibility = True
data_path = \\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\Amazon_Sports_and_Outdoors
show_progress = True
save_dataset = False
save_dataloaders = False
benchmark_filename = None

Training Hyper Parameters:
checkpoint_dir = saved
epochs = 200
train_batch_size = 1024
learner = adam
learning_rate = 0.0001
eval_step = 2
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'group_by': 'user', 'order': 'TO', 'mode': 'full'}
metrics = ['Recall', 'NDCG']
topk = [3, 5, 10, 20]
valid_metric = Recall@10
valid_metric_bigger = True
eval_batch_size = 1
metric_decimal_place = 4

Dataset Hyper Parameters:
field_separator = 	
seq_separator =  
USER_ID_FIELD = user_id
ITEM_ID_FIELD = item_id
RATING_FIELD = rating
TIME_FIELD = timestamp
seq

parameters = {'General': ['gpu_id', 'use_gpu', 'seed', 'reproducibility', 'state', 'data_path', 'benchmark_filename', 'show_progress', 'config_file', 'save_dataset', 'save_dataloaders'], 'Training': ['epochs', 'train_batch_size', 'learner', 'learning_rate', 'training_neg_sample_num', 'training_neg_sample_distribution', 'eval_step', 'stopping_step', 'checkpoint_dir', 'clip_grad_norm', 'loss_decimal_place', 'weight_decay'], 'Evaluation': ['eval_args', 'metrics', 'topk', 'valid_metric', 'valid_metric_bigger', 'eval_batch_size', 'metric_decimal_place'], 'Dataset': ['field_separator', 'seq_separator', 'USER_ID_FIELD', 'ITEM_ID_FIELD', 'RATING_FIELD', 'TIME_FIELD', 'seq_len', 'LABEL_FIELD', 'threshold', 'NEG_PREFIX', 'ITEM_LIST_LENGTH_FIELD', 'LIST_SUFFIX', 'MAX_ITEM_LIST_LENGTH', 'POSITION_FIELD', 'HEAD_ENTITY_ID_FIELD', 'TAIL_ENTITY_ID_FIELD', 'RELATION_ID_FIELD', 'ENTITY_ID_FIELD', 'load_col', 'unload_col', 'unused_col', 'additional_feat_suffix', 'filter_inter_by_user_or_item', 'rm_dup_in

\\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\dataset.py:446: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value='', inplace=True)
\\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\dataset.py:446: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on wh

parameters = {'General': ['gpu_id', 'use_gpu', 'seed', 'reproducibility', 'state', 'data_path', 'benchmark_filename', 'show_progress', 'config_file', 'save_dataset', 'save_dataloaders'], 'Training': ['epochs', 'train_batch_size', 'learner', 'learning_rate', 'training_neg_sample_num', 'training_neg_sample_distribution', 'eval_step', 'stopping_step', 'checkpoint_dir', 'clip_grad_norm', 'loss_decimal_place', 'weight_decay'], 'Evaluation': ['eval_args', 'metrics', 'topk', 'valid_metric', 'valid_metric_bigger', 'eval_batch_size', 'metric_decimal_place'], 'Dataset': ['field_separator', 'seq_separator', 'USER_ID_FIELD', 'ITEM_ID_FIELD', 'RATING_FIELD', 'TIME_FIELD', 'seq_len', 'LABEL_FIELD', 'threshold', 'NEG_PREFIX', 'ITEM_LIST_LENGTH_FIELD', 'LIST_SUFFIX', 'MAX_ITEM_LIST_LENGTH', 'POSITION_FIELD', 'HEAD_ENTITY_ID_FIELD', 'TAIL_ENTITY_ID_FIELD', 'RELATION_ID_FIELD', 'ENTITY_ID_FIELD', 'load_col', 'unload_col', 'unused_col', 'additional_feat_suffix', 'filter_inter_by_user_or_item', 'rm_dup_in

25 Jan 20:15    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 212
state = INFO
reproducibility = True
data_path = \\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\Amazon_Sports_and_Outdoors
show_progress = True
save_dataset = False
save_dataloaders = False
benchmark_filename = None

Training Hyper Parameters:
checkpoint_dir = saved
epochs = 200
train_batch_size = 1024
learner = adam
learning_rate = 0.0001
eval_step = 2
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'group_by': 'user', 'order': 'TO', 'mode': 'full'}
metrics = ['Recall', 'NDCG']
topk = [3, 5, 10, 20]
valid_metric = Recall@10
valid_metric_bigger = True
eval_batch_size = 1
metric_decimal_place = 4

Dataset Hyper Parameters:
field_separator = 	
seq_separator =  
USER_ID_FIELD = user_id
ITEM_ID_FIELD = item_id
RATING_FIELD = rating
TIME_FIELD = timestamp
seq

parameters = {'General': ['gpu_id', 'use_gpu', 'seed', 'reproducibility', 'state', 'data_path', 'benchmark_filename', 'show_progress', 'config_file', 'save_dataset', 'save_dataloaders'], 'Training': ['epochs', 'train_batch_size', 'learner', 'learning_rate', 'training_neg_sample_num', 'training_neg_sample_distribution', 'eval_step', 'stopping_step', 'checkpoint_dir', 'clip_grad_norm', 'loss_decimal_place', 'weight_decay'], 'Evaluation': ['eval_args', 'metrics', 'topk', 'valid_metric', 'valid_metric_bigger', 'eval_batch_size', 'metric_decimal_place'], 'Dataset': ['field_separator', 'seq_separator', 'USER_ID_FIELD', 'ITEM_ID_FIELD', 'RATING_FIELD', 'TIME_FIELD', 'seq_len', 'LABEL_FIELD', 'threshold', 'NEG_PREFIX', 'ITEM_LIST_LENGTH_FIELD', 'LIST_SUFFIX', 'MAX_ITEM_LIST_LENGTH', 'POSITION_FIELD', 'HEAD_ENTITY_ID_FIELD', 'TAIL_ENTITY_ID_FIELD', 'RELATION_ID_FIELD', 'ENTITY_ID_FIELD', 'load_col', 'unload_col', 'unused_col', 'additional_feat_suffix', 'filter_inter_by_user_or_item', 'rm_dup_in

25 Jan 20:15    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 212
state = INFO
reproducibility = True
data_path = \\wsl.localhost\Ubuntu\home\piotrs\Uni\SequentialRecommendation\recbole\data\dataset\Amazon_Sports_and_Outdoors
show_progress = True
save_dataset = False
save_dataloaders = False
benchmark_filename = None

Training Hyper Parameters:
checkpoint_dir = saved
epochs = 200
train_batch_size = 1024
learner = adam
learning_rate = 0.0001
eval_step = 2
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'group_by': 'user', 'order': 'TO', 'mode': 'full'}
metrics = ['Recall', 'NDCG']
topk = [3, 5, 10, 20]
valid_metric = Recall@10
valid_metric_bigger = True
eval_batch_size = 1
metric_decimal_place = 4

Dataset Hyper Parameters:
field_separator = 	
seq_separator =  
USER_ID_FIELD = user_id
ITEM_ID_FIELD = item_id
RATING_FIELD = rating
TIME_FIELD = timestamp
seq

In [4]:
def predict_item(inp_model, inp_interaction):
    item_seq = inp_interaction[inp_model.ITEM_SEQ]
    item_seq_len = inp_interaction[inp_model.ITEM_SEQ_LEN]

    seq_output = inp_model.forward(item_seq, item_seq_len)

    test_items_emb = inp_model.item_embedding.weight
    scores = torch.matmul(seq_output, test_items_emb.transpose(0, 1))

    idx = torch.argmax(scores, dim=1)

    return idx

### Import mappings

In [5]:
df_items = pd.read_csv('../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.item', sep='\t')
df_items.head(5)

,item_id:token,title:token,price:float,brand:token,categories:token_seq,sales_type:token,sales_rank:float
0,0,Ghost Inc Glock Armorers Tool 3/32 Punch,9.99,Ghost,"'Sports & Outdoors', 'Hunting', 'Hunting & Fis...",Sports &amp; Outdoors,532941.0
1,1,5 LED Bicycle Rear Tail Red Bike Torch Laser B...,8.26,NaN,"'Lights & Reflectors', 'Sports & Outdoors', 'C...",Toys & Games,15617.0
2,3,Black Mountain Products Resistance Band Set wi...,32.99,Black Mountain,"'Accessories', 'Exercise Bands', 'Sports & Out...",Sports &amp; Outdoors,1010.0
3,2,Black Mountain Products Single Resistance Band...,10.49,Black Mountain,"'Accessories', 'Exercise Bands', 'Sports & Out...",Sports &amp; Outdoors,1010.0
4,4,Outers Universal 32-Piece Blow Molded Gun Clea...,21.99,Outers,"'Sports & Outdoors', 'Hunting', 'Gun Cleaning ...",Sports &amp; Outdoors,26457.0


In [6]:
data = []
with open('../../Amazon_Sports_and_Outdoors/meta_Sports_and_Outdoors.json', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(ast.literal_eval(line))  # safe Python dict parser

df_meta = pd.DataFrame(data)
df_meta.head(5)

,asin,title,price,imUrl,related,brand,categories,salesRank,description
0,0000032069,Adult Ballet Tutu Cheetah Pink,7.89,http://ecx.images-amazon.com/images/I/51EzU6qu...,"{'also_bought': ['0000032050', 'B00D0DJAEG', '...",BubuBibi,"[[Sports & Outdoors, Other Sports, Dance, Clot...",NaN,NaN
1,0000031909,Girls Ballet Tutu Neon Pink,7.00,http://ecx.images-amazon.com/images/I/41xBoP0F...,"{'also_bought': ['B002BZX8Z6', 'B00JHONN1S', '...",Unknown,"[[Sports & Outdoors, Other Sports, Dance]]",{'Toys & Games': 201847},High quality 3 layer ballet tutu. 12 inches in...
2,0000032034,Adult Ballet Tutu Yellow,7.87,http://ecx.images-amazon.com/images/I/21GNUNIa...,"{'also_bought': ['B00D2JSRFQ', '0000032042', '...",BubuBibi,"[[Sports & Outdoors, Other Sports, Dance, Clot...",NaN,NaN
3,0000031852,Girls Ballet Tutu Zebra Hot Pink,3.17,http://ecx.images-amazon.com/images/I/51fAmVkT...,"{'also_bought': ['B00JHONN1S', 'B002BZX8Z6', '...",Coxlures,"[[Sports & Outdoors, Other Sports, Dance]]",{'Toys & Games': 211836},TUtu
4,0000032050,Adult Ballet Tutu Purple,12.85,http://ecx.images-amazon.com/images/I/41TxNYG8...,"{'also_bought': ['B00D2JSRFQ', 'B00D2JTMS2', '...",BubuBibi,"[[Sports & Outdoors, Other Sports, Dance, Clot...",NaN,NaN


In [7]:
with open('../../Amazon_Sports_and_Outdoors/item_mapping_Amazon_Sports_and_Outdoors.json', 'r') as f:
    item_mapping = json.load(f)

print(item_mapping[:10])

['1881509818', '2094869245', '7245456259', '7245456313', 'B000002NUS', 'B00000ELZ5', 'B00000IURU', 'B00000IUX5', 'B00000J6JO', 'B0000224UE']


In [12]:
pd.set_option('display.max_colwidth', None)

for interaction, _, _, _ in test_data:
    asins = set()
    for model in models:
        index = predict_item(model, interaction)
        index = index[0].item()
        asin = item_mapping[index]

        asins.add(asin)

    if len(asins) != len(models):
        continue

    print("-------- GROUND-TRUTH --------")
    pos_item = interaction[models[0].POS_ITEM_ID].item()
    asin = item_mapping[pos_item]
    row = df_meta[df_meta["asin"] == asin]

    title = row["title"].item()
    categories = row["categories"].item()
    imUrl = row["imUrl"].item()

    print(f'Title: {title} \n Categories: {categories} \n imUrl: {imUrl}')

    for i, model in enumerate(models):
        print(f"-------- PREDICTION {i} --------")
        index = predict_item(model, interaction)
        index = index[0].item()
        asin = item_mapping[index]

        row = df_meta[df_meta["asin"] == asin]

        title = row["title"].item()
        categories = row["categories"].item()
        imUrl = row["imUrl"].item()

        print(f'Title: {title} \n Categories: {categories} \n imUrl: {imUrl}')

-------- GROUND-TRUTH --------
Title: Pedro's Universal Bicycle Crank Remover with Handle 
 Categories: [['Sports & Outdoors', 'Cycling', 'Bike Tools & Maintenance', 'Shop Tools']] 
 imUrl: http://ecx.images-amazon.com/images/I/31Z25FD60EL._SY300_.jpg
-------- PREDICTION 0 --------
Title: Park Tool Spoke Wrench SW-0 Through SW-3 
 Categories: [['Sports & Outdoors', 'Cycling', 'Bike Tools & Maintenance', 'Shop Tools']] 
 imUrl: http://ecx.images-amazon.com/images/I/31bu1jfVQxL._SY300_.jpg
-------- PREDICTION 1 --------
Title: Sawyer Products PointOne Squeeze Water Filtration System 
 Categories: [['Sports & Outdoors', 'Outdoor Gear', 'Camping & Hiking', 'Hydration', 'Water Filters']] 
 imUrl: http://ecx.images-amazon.com/images/I/51SJ5lM%2BhyL._SY300_.jpg
-------- PREDICTION 2 --------
Title: Pearl iZUMi Men's Quest Cycling Bib Short,Black,X-Large 
 Categories: [['Sports & Outdoors', 'Clothing', 'Men', 'Shorts', 'Compression Shorts']] 
 imUrl: http://ecx.images-amazon.com/images/I/315Sf